This notebook was made to answer sub question 4 of this research project.

In [6]:
import pandas as pd

columns = [
    "verkeersongeval_nummer",
    "partij_1_objecttype",
    "partij_1_objecttype_overig",
    "partij_2_objecttype",
    "partij_2_objecttype_overig",
    "maximum_snelheid",
    "verkeersongeval_afloop",
]

ongevallen_df = pd.read_csv("ongevallen_2024_amsterdam.csv", usecols=columns)
ongevallen_df.head()

party_columns = [
    "partij_1_objecttype",
    "partij_1_objecttype_overig",
    "partij_2_objecttype",
    "partij_2_objecttype_overig",
]

for column in party_columns:
    print(f"\nCounts for {column}:")
    print(ongevallen_df[column].value_counts(dropna=False))


Counts for partij_1_objecttype:
partij_1_objecttype
Personenauto                           3028
NaN                                    1111
Fiets                                   590
Bromfiets                               348
Bestelauto                              341
Motor                                   217
Snorfiets                               159
Overig wegmeubilair                     117
Lichtmast                                64
Bus                                      48
Vrachtauto                               48
Brommobiel                               32
Trein/tram                               25
Trekker                                  20
Boom                                     16
e-bike                                   15
Scootmobiel                              11
Voetganger                                2
Landbouwvoertuig                          2
Onbekend voertuig i.g.v. doorrijder       1
Overig vast object                        1
Name: count, dtype: int

In [7]:
speed_limits = pd.to_numeric(ongevallen_df["maximum_snelheid"], errors="coerce")
accident_counts = speed_limits.value_counts().reindex([30, 50], fill_value=0).astype(int)

print(f"Accidents on 30 km/h roads: {accident_counts[30]}")
print(f"Accidents on 50 km/h roads: {accident_counts[50]}")

Accidents on 30 km/h roads: 2576
Accidents on 50 km/h roads: 1013


In [10]:
vehicle_records = pd.concat(
    [
        ongevallen_df[
            [
                "verkeersongeval_nummer",
                "maximum_snelheid",
                "partij_1_objecttype",
                "partij_1_objecttype_overig",
            ]
        ].rename(
            columns={
                "partij_1_objecttype": "voertuigtype",
                "partij_1_objecttype_overig": "voertuigtype_overig",
            }
        ),
        ongevallen_df[
            [
                "verkeersongeval_nummer",
                "maximum_snelheid",
                "partij_2_objecttype",
                "partij_2_objecttype_overig",
            ]
        ].rename(
            columns={
                "partij_2_objecttype": "voertuigtype",
                "partij_2_objecttype_overig": "voertuigtype_overig",
            }
        ),
    ],
    ignore_index=True,
)

vehicle_records["maximum_snelheid"] = pd.to_numeric(
    vehicle_records["maximum_snelheid"], errors="coerce"
)
vehicle_records["voertuigtype"] = vehicle_records["voertuigtype"].fillna("Onbekend")
overig = vehicle_records["voertuigtype_overig"].fillna("").astype(str).str.strip()
vehicle_records["voertuigtype"] = (
    vehicle_records["voertuigtype"].astype(str)
    + overig.where(overig.eq(""), ": " + overig)
)

vehicle_records = vehicle_records[
    vehicle_records["maximum_snelheid"].isin([30, 50])
].drop_duplicates(
    subset=["verkeersongeval_nummer", "maximum_snelheid", "voertuigtype"]
)

accidents_by_vehicle_speed = (
    vehicle_records.groupby(["voertuigtype", "maximum_snelheid"])[
        "verkeersongeval_nummer"
    ]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=[30, 50], fill_value=0)
)
accidents_by_vehicle_speed.columns = ["30 km/h", "50 km/h"]
accidents_by_vehicle_speed["Total"] = accidents_by_vehicle_speed.sum(axis=1)
accidents_by_vehicle_speed = accidents_by_vehicle_speed.sort_values(
    by="Total", ascending=False
)
print(accidents_by_vehicle_speed.to_string())

                                     30 km/h  50 km/h  Total
voertuigtype                                                
Personenauto                            1596      781   2377
Fiets                                    972      194   1166
Bromfiets                                398      106    504
Onbekend                                 332       89    421
Bestelauto                               258      114    372
Motor                                    182       64    246
Snorfiets                                182       25    207
Voetganger                               152       37    189
Onbekend: PARTIJ ONTBREEKT                93       19    112
Bus                                       41       32     73
Lichtmast                                 26       42     68
Overig vast object                        46       19     65
Vrachtauto                                38       26     64
Trein/tram                                37       11     48
Overig wegmeubilair     